# Explorative Data Analysis (EDA) of a Laser Cutting Process - Solutions

In this exercise, we perform an explorative data analysis (EDA) on quality data of a **laser cutting process for metal sheets** - the same kind of data used as example in the lecture.

Each row is one cut metal-sheet part. The most important quality attribute is the **burr height** (German: *Grathöhe*) in micrometer (``Gth_micro``). It should be as small as possible: a lower burr height corresponds to a better cutting quality.

The exercise follows the six EDA steps introduced in the lecture.

Questions in the tasks are meant for thinking - you do not have to write down your answers. Tasks marked with **Experiment** ask you to change your code and to observe the effect.

Tasks marked with **Optional** introduce further techniques. Skip them if you are short on time - they are not needed for the following tasks, but you can come back to them later, e.g., as a reference for your own EDA.

Do not be shy of using the [pandas](https://pandas.pydata.org/docs/) and [seaborn](https://seaborn.pydata.org/api.html) documentation!

### Contents <a id="contents"></a>
1. [Dataset Understanding](#1-dataset-understanding)
    * [Dataset Documentation](#dataset-documentation)
    * [1.1 Attribute Overview](#1-1-attribute-overview)
    * [1.2 Data Quality Checks](#1-2-data-quality-checks)
2. [Univariate Analysis](#2-univariate-analysis)
    * [2.1 Summary Statistics](#2-1-summary-statistics)
    * [2.2 Visualize Distributions](#2-2-visualize-distributions)
    * [2.3 Handling Missing Values](#2-3-handling-missing-values)
3. [Bivariate and Multivariate Analysis](#3-bivariate-and-multivariate-analysis)
    * [3.1 Bivariate Analysis](#3-1-bivariate-analysis)
    * [3.2 Multivariate Analysis](#3-2-multivariate-analysis)
4. [Outlier Detection](#4-outlier-detection)
5. [Dealing with High-Dimensional Data](#5-dealing-with-high-dimensional-data)
6. [Summary and Insights](#6-summary-and-insights)

You will apply the pandas and seaborn skills from the tutorials. In addition, we use the library ``missingno`` to visualize missing values. If it is not installed yet, please execute the code cell below.

In [ ]:
# You might have to install missingno first
%pip install missingno

In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
from matplotlib import pyplot as plt
import missingno as msno

## 1. Dataset Understanding <a id="1-dataset-understanding"></a>

[Back to top](#contents)

Get an overview of the attributes and perform initial sanity checks.

### Dataset Documentation <a id="dataset-documentation"></a>

The data can be found in the directory ``4_data`` (file ``laser_cutting.csv``). Each row is a laser-cut metal sheet. The following descriptions were provided by the production department:

* ``Id``: identifier of the cut part.
* ``Plant``: production plant.
* ``Machine``: laser cutting machine used.
* ``Material``: sheet material (steel, aluminum or copper).
* ``SurfaceQuality``: surface quality class of the raw sheet, from ``A`` (best) to ``C`` (worst).
* ``Thickness_mm``: sheet thickness in mm.
* ``LaserPower_W``: laser power in W.
* ``CuttingSpeed_mm_s``: cutting speed in mm/s.
* ``GasPressure_bar``: cutting gas pressure in bar.
* ``Temperature_C``: temperature measured by a pyrometer at the cutting head in °C.
* ``ProductionTime``: production time of the part.
* ``Gth_micro``: Grathöhe (burr height) in micrometer - our quality target.
* ``NeedsDeburring``: whether the part has to be deburred after cutting.

### 1.1 Attribute Overview <a id="1-1-attribute-overview"></a>

Read ``laser_cutting.csv`` into a ``DataFrame`` named ``laser`` and print its ``shape``.
Inspect the first **and** last rows (``head()``, ``tail()``) and use ``info()`` to get an overview of the columns and their data types.

In [ ]:
data_path = '../4_data/laser_cutting.csv'
### BEGIN SOLUTION
laser = pd.read_csv(data_path)
print(laser.shape)
display(laser.head())
laser.tail()
### END SOLUTION

In [ ]:
### BEGIN SOLUTION
laser.info()
### END SOLUTION

For each attribute, determine its **scale type** (nominal, ordinal, discrete cardinal, continuous cardinal) and its **unit of measurement** (compare the attribute overview slide of the lecture).
Which attribute names or units are unclear? Which questions would you ask the production department?

#### Explanation

| Attribute | Scale type | Unit |
|---|---|---|
| ``Id`` | identifier (nominal) | - |
| ``Plant`` | nominal | - |
| ``Machine`` | nominal | - |
| ``Material`` | nominal | - |
| ``SurfaceQuality`` | ordinal (A > B > C) | - |
| ``Thickness_mm`` | continuous cardinal | mm |
| ``LaserPower_W`` | continuous cardinal | W |
| ``CuttingSpeed_mm_s`` | continuous cardinal | mm/s |
| ``GasPressure_bar`` | continuous cardinal | bar |
| ``Temperature_C`` | continuous cardinal | °C |
| ``ProductionTime`` | continuous cardinal | unknown (seconds?) |
| ``Gth_micro`` | continuous cardinal | µm |
| ``NeedsDeburring`` | nominal (binary: Yes / No) | - |

Open questions for the production department:
* ``ProductionTime``: Which unit? Does it include handling or only the cutting time?
* ``Gth_micro``: an abbreviation of the German term *Grathöhe* - a name like ``BurrHeight_um`` would be clearer. How and where is the burr height measured?
* ``Temperature_C``: Where exactly is the temperature measured?
* ``NeedsDeburring``: Who decides this, and based on which criterion?

### 1.2 Data Quality Checks <a id="1-2-data-quality-checks"></a>

Compute descriptive statistics with ``describe()`` and check the domain (min to max) of each numerical attribute.
Do the minimum and maximum values make sense given the units of measurement?

In [ ]:
### BEGIN SOLUTION
laser.describe()
### END SOLUTION

#### Explanation

* ``Temperature_C``: the maximum of 999 °C is suspicious - 75 % of the values are below ~830 °C. A round value like 999 is a typical code for missing values (*Missing Values Trap*).
* ``Gth_micro``: a minimum of 0.02 µm is implausible, as 75 % of the parts have a burr height above ~22 µm.
* ``GasPressure_bar``: the minimum of 1.8 bar is far below the 25 % quantile of ~7 bar.
* ``ProductionTime``: the maximum (~125) is much larger than the 75 % quantile (~43).

Keep these observations in mind for the outlier detection (Step 4).

For the categorical attributes (``Plant``, ``Machine``, ``Material``, ``SurfaceQuality``, ``NeedsDeburring``), check the set of categories and their frequencies with ``value_counts()``.

In [ ]:
cat_cols = ['Plant', 'Machine', 'Material', 'SurfaceQuality', 'NeedsDeburring']
### BEGIN SOLUTION
for col in cat_cols:
    display(laser[col].value_counts(dropna=False))
### END SOLUTION

Check how many missing values each attribute has (``isna()``) and visualize them with the function ``matrix()`` from the library ``missingno`` (imported as ``msno``), as shown in the lecture.

In [ ]:
### BEGIN SOLUTION
print(laser.isna().sum())
msno.matrix(laser)
plt.show()
### END SOLUTION

Replace the coded missing values ``999`` in ``Temperature_C`` by ``NaN`` (``.replace(999, np.nan)``).
Recount the missing values and verify that the maximum temperature is now in a plausible range.

In [ ]:
### BEGIN SOLUTION
laser['Temperature_C'] = laser['Temperature_C'].replace(999, np.nan)
print('Missing temperature values:', laser['Temperature_C'].isna().sum())
print('Max temperature:', laser['Temperature_C'].max())
### END SOLUTION

Finally, check whether the dataset contains duplicate rows (``duplicated()``) and whether ``Id`` is a unique identifier (*Hint:* attribute ``is_unique`` of a ``Series``).
Display all duplicated rows (*Hint:* parameter ``keep`` of ``duplicated()``) and remove the duplicates (``drop_duplicates()``).

In [ ]:
### BEGIN SOLUTION
print('Duplicate rows:', laser.duplicated().sum())
print('Id unique:', laser['Id'].is_unique)
display(laser[laser.duplicated(keep=False)])
laser = laser.drop_duplicates()
print(laser.shape)
### END SOLUTION

## 2. Univariate Analysis <a id="2-univariate-analysis"></a>

[Back to top](#contents)

Understand the distribution and variability of single attributes.

### 2.1 Summary Statistics <a id="2-1-summary-statistics"></a>

For the attributes ``Gth_micro`` and ``ProductionTime``, compute the mean, median, standard deviation, interquartile range (IQR), coefficient of variation (CV) and skewness (``skew()``).

*Hint:* ``quantile([0.25, 0.75])`` returns the 25 % and 75 % quantiles (Q1 and Q3); the IQR is Q3 - Q1.

**Experiment:** Remove the 5 longest production times (*Hint:* ``sort_values()``) and recompute mean and median of ``ProductionTime``. Which of the two changes more?

In [ ]:
### BEGIN SOLUTION
for col in ['Gth_micro', 'ProductionTime']:
    x = laser[col]
    q1, q3 = x.quantile([0.25, 0.75])
    print(col)
    print(f'  Mean:     {x.mean():.2f}')
    print(f'  Median:   {x.median():.2f}')
    print(f'  Std:      {x.std():.2f}')
    print(f'  IQR:      {q3 - q1:.2f}')
    print(f'  CV:       {x.std() / x.mean():.2f}')
    print(f'  Skewness: {x.skew():.2f}')
### END SOLUTION

In [ ]:
### BEGIN SOLUTION
production_time = laser['ProductionTime'].dropna()
production_time_reduced = production_time.sort_values().iloc[:-5]
print(f'Mean:   {production_time.mean():.2f} -> {production_time_reduced.mean():.2f}')
print(f'Median: {production_time.median():.2f} -> {production_time_reduced.median():.2f}')
### END SOLUTION

#### Explanation

``Gth_micro`` is roughly symmetric (mean and median almost equal), ``ProductionTime`` is right-skewed (mean > median, large positive skewness).

Removing only the 5 longest production times lowers the mean by about 1.3, but the median by only about 0.4: the median is **robust** against extreme values, the mean is not.

### 2.2 Visualize Distributions <a id="2-2-visualize-distributions"></a>

Create bar charts (``countplot()``) for all categorical attributes in ``cat_cols`` using a loop.
Additionally, show the material frequencies as relative frequencies (*Hint:* parameter ``stat`` of ``countplot()``) and check for imbalances (compare the lecture).

In [ ]:
### BEGIN SOLUTION
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
for ax, col in zip(axes.flat, cat_cols):
    sns.countplot(laser, x=col, ax=ax)
sns.countplot(laser, x='Material', stat='proportion', ax=axes.flat[5])
axes.flat[5].set_title('Material (relative)')
plt.tight_layout()
plt.show()
### END SOLUTION

Create a histogram (``sns.histplot()``) of the target ``Gth_micro`` and add a kernel density estimation (``kde=True``).

**Experiment:** Change the number of bins (parameter ``bins``, e.g., 5, 20, 60). With which numbers of bins do you see the isolated values near 0 µm and above 45 µm?

In [ ]:
### BEGIN SOLUTION
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, bins in zip(axes, [5, 20, 60]):
    sns.histplot(laser, x='Gth_micro', bins=bins, kde=True, ax=ax)
    ax.set_title(f'Burr Height (um) - {bins} bins')
plt.tight_layout()
plt.show()
### END SOLUTION

#### Explanation

Most parts have a burr height between ~15 and ~40 µm (roughly bell-shaped). With enough bins (e.g., 20 or 60), a few isolated values become visible: some at almost 0 µm and some at 45-57 µm.
With only 5 bins, these values are hidden in the outer bars - the number of bins matters!

Plot histograms for all numerical process attributes (listed in ``num_cols``) using a loop.

In [ ]:
num_cols = ['Thickness_mm', 'LaserPower_W', 'CuttingSpeed_mm_s',
            'GasPressure_bar', 'Temperature_C', 'ProductionTime']
### BEGIN SOLUTION
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
for ax, col in zip(axes.flat, num_cols):
    sns.histplot(laser, x=col, kde=True, ax=ax)
plt.tight_layout()
plt.show()
### END SOLUTION

### 2.3 Handling Missing Values <a id="2-3-handling-missing-values"></a>

Compute the share of missing values per attribute (in %).
For ``Temperature_C``, check whether the values are missing randomly or systematically: compute the share of missing temperature values per ``Machine``.

Decide how to handle the missing values (see the handling strategies in the lecture) and justify your decision.

*Hint:* ``dropna(subset=[...])`` removes all rows that have missing values in the given columns.

In [ ]:
### BEGIN SOLUTION
display((laser.isna().mean() * 100).round(1))
display(laser.groupby('Machine')['Temperature_C'].apply(lambda s: s.isna().mean() * 100))
# rows without target value cannot be used
laser = laser.dropna(subset=['Gth_micro'])
### END SOLUTION

#### Explanation

``Temperature_C`` is missing **systematically**: machine ``m_d_001`` has no temperature values at all (it probably has no pyrometer), whereas only ~4 % (the coded 999 values) are missing for ``m_si_009``. The other attributes have ~2 % randomly missing values.

Decisions:
* ``Gth_micro`` is our target: rows without target value cannot be used, so they are removed.
* ``Temperature_C``: imputation for an entire machine is not meaningful. Keep it for now and decide in Step 5 whether to use it as a feature.
* Other attributes: keep the rows; impute missing values (e.g., median / mode) in the modeling phase.

## 3. Bivariate and Multivariate Analysis <a id="3-bivariate-and-multivariate-analysis"></a>

[Back to top](#contents)

Investigate relationships between attributes - especially between the process parameters and the quality target ``Gth_micro``.

### 3.1 Bivariate Analysis <a id="3-1-bivariate-analysis"></a>

Create scatter plots of ``Gth_micro`` vs. ``Thickness_mm``, ``LaserPower_W``, ``CuttingSpeed_mm_s`` and ``GasPressure_bar``, colored by ``Material`` (parameter ``hue``).

**Experiment:** Change ``hue`` to ``Machine``, ``SurfaceQuality`` and ``NeedsDeburring``. Which attribute separates the points most clearly?

In [ ]:
### BEGIN SOLUTION
fig, axes = plt.subplots(2, 2, figsize=(12, 9))
for ax, col in zip(axes.flat, ['Thickness_mm', 'LaserPower_W', 'CuttingSpeed_mm_s', 'GasPressure_bar']):
    sns.scatterplot(laser, x=col, y='Gth_micro', hue='Material', ax=ax)
plt.tight_layout()
plt.show()
### END SOLUTION

In [ ]:
### BEGIN SOLUTION
fig, axes = plt.subplots(2, 2, figsize=(12, 9))
for ax, col in zip(axes.flat, ['Thickness_mm', 'LaserPower_W', 'CuttingSpeed_mm_s', 'GasPressure_bar']):
    sns.scatterplot(laser, x=col, y='Gth_micro', hue='NeedsDeburring', ax=ax)
plt.tight_layout()
plt.show()
### END SOLUTION

#### Explanation

The burr height tends to increase with sheet thickness, laser power and cutting speed and to decrease with gas pressure. All relationships are noisy. The parts cut with ~2 bar gas pressure have very high burr heights, and a few parts have a burr height of almost 0 µm.

``Machine`` and ``SurfaceQuality`` do not separate the points. ``NeedsDeburring`` separates them perfectly by a horizontal line at ~30 µm (apart from the points near 0 µm) - we will come back to this in Step 5.

**Optional:** The seaborn function ``lmplot()`` combines a scatter plot with a fitted linear regression line - optionally for each group of a categorical attribute (named parameters ``hue``, ``col`` or ``row``).

Use it to plot ``Gth_micro`` vs. ``CuttingSpeed_mm_s`` with a separate regression line for each ``Material``.

In [ ]:
### BEGIN SOLUTION
sns.lmplot(laser, x='CuttingSpeed_mm_s', y='Gth_micro', hue='Material', markers='.')
plt.show()
### END SOLUTION

Create a scatter plot of ``CuttingSpeed_mm_s`` vs. ``LaserPower_W``, colored by ``Material`` (compare the lecture).
How are the two process parameters related? What does this mean for the interpretation of the previous plots?

In [ ]:
### BEGIN SOLUTION
sns.scatterplot(laser, x='LaserPower_W', y='CuttingSpeed_mm_s', hue='Material')
plt.title('Cutting Speed vs. Laser Power by Material')
plt.show()
### END SOLUTION

#### Explanation

The cutting speed increases with the laser power: the operators cut faster when more power is available. Aluminum is cut fastest, copper slowest.

Interpretation: as both parameters are related, we cannot tell from the plots alone whether the laser power itself increases the burr height, or whether it only appears so because high power goes along with high cutting speed (correlation is not causation!).

Create grouped box plots (``sns.boxplot()``) of ``Gth_micro`` by ``Material`` - once without and once with ``SurfaceQuality`` as ``hue`` (compare the lecture) - and of ``Gth_micro`` by ``NeedsDeburring``.

In [ ]:
### BEGIN SOLUTION
fig, axes = plt.subplots(1, 3, figsize=(18, 4))
sns.boxplot(laser, x='Material', y='Gth_micro', ax=axes[0])
sns.boxplot(laser, x='Material', y='Gth_micro', hue='SurfaceQuality', hue_order=['A', 'B', 'C'], ax=axes[1])
sns.boxplot(laser, x='NeedsDeburring', y='Gth_micro', ax=axes[2])
plt.tight_layout()
plt.show()
### END SOLUTION

As the materials are imbalanced (only few copper parts), their burr height distributions are hard to compare in a histogram with absolute counts.

Create two histograms of ``Gth_micro`` with ``hue='Material'``: one with default settings and one using the named parameters ``stat='proportion'`` and ``common_norm=False``, so that the histogram of each material is normalized on its own.

In [ ]:
### BEGIN SOLUTION
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
sns.histplot(laser, x='Gth_micro', hue='Material', ax=axes[0])
axes[0].set_title('Absolute counts')
sns.histplot(laser, x='Gth_micro', hue='Material', stat='proportion', common_norm=False, ax=axes[1])
axes[1].set_title('Proportions per material')
plt.tight_layout()
plt.show()
### END SOLUTION

Statistical values per group complement the plots. Use ``groupby()`` and ``agg()`` to compute the mean, median, minimum, maximum and count of ``Gth_micro`` per ``Material`` with a single function call.

In [ ]:
### BEGIN SOLUTION
laser.groupby('Material')['Gth_micro'].agg(['mean', 'median', 'min', 'max', 'count']).round(1)
### END SOLUTION

Compute the Pearson correlation coefficients between ``Gth_micro`` and the numerical attributes (``num_cols``).
Then compute the Spearman rank correlation as well (``corr(method='spearman')``).

In [ ]:
### BEGIN SOLUTION
cols = num_cols + ['Gth_micro']
pd.DataFrame({
    'pearson': laser[cols].corr()['Gth_micro'],
    'spearman': laser[cols].corr(method='spearman')['Gth_micro'],
}).round(2)
### END SOLUTION

### 3.2 Multivariate Analysis <a id="3-2-multivariate-analysis"></a>

Create correlation matrices of the numerical attributes in ``num_cols`` and the target using **both** methods (Pearson and Spearman) and display them as heatmaps (``sns.heatmap()``) side by side (*Hint:* ``plt.subplots()`` and the parameter ``ax``, see the seaborn tutorial).

Identifiers and nominal attributes must not be included, even if they are stored as numbers.

Which pairs of attributes are (pairwise) collinear (``|r| > 0.7``), and for which pair do Pearson and Spearman differ most? Why?

In [ ]:
### BEGIN SOLUTION
fig, axes = plt.subplots(1, 2, figsize=(18, 7))
for ax, method in zip(axes, ['pearson', 'spearman']):
    sns.heatmap(laser[cols].corr(method=method), annot=True, fmt='.2f',
                cmap='coolwarm', vmin=-1, vmax=1, ax=ax)
    ax.set_title(f'Correlation Matrix ({method})')
plt.tight_layout()
plt.show()
### END SOLUTION

#### Explanation

* **Collinear:** ``CuttingSpeed_mm_s`` and ``ProductionTime`` (Pearson ~ -0.8, Spearman ~ -0.9): the faster the cut, the shorter the production time. Laser power is moderately correlated with cutting speed and temperature (~0.6-0.65).
* **Largest difference** between Pearson and Spearman: ``CuttingSpeed_mm_s`` vs. ``ProductionTime``. The relationship is monotonic but not linear (time ~ length / speed), so Spearman captures it better than Pearson.

Create a pair plot (``sns.pairplot()``, parameter ``vars`` for the selection of columns) of ``Thickness_mm``, ``CuttingSpeed_mm_s``, ``GasPressure_bar``, ``ProductionTime`` and ``Gth_micro``, colored by ``NeedsDeburring`` (compare the lecture).

In [ ]:
### BEGIN SOLUTION
sns.pairplot(laser, vars=['Thickness_mm', 'CuttingSpeed_mm_s', 'GasPressure_bar', 'ProductionTime', 'Gth_micro'],
             hue='NeedsDeburring')
plt.show()
### END SOLUTION

**Optional:** Investigate the combined influence of two categorical attributes, ``Material`` and ``Machine``, on the burr height.

Compute the mean burr height for each combination using ``groupby()`` with both columns, and transform the result into a table using the ``Series`` method ``unstack()``.
Visualize the table with the pandas plotting method ``plot(kind='bar')``.

In [ ]:
### BEGIN SOLUTION
gth_material_machine = laser.groupby(['Material', 'Machine'])['Gth_micro'].mean().unstack()
display(gth_material_machine.round(1))
gth_material_machine.plot(kind='bar')
plt.ylabel('Mean burr height (um)')
plt.show()
### END SOLUTION

**Optional:** Seaborn's category plot (``catplot()``) visualizes the combined influence of categorical attributes directly from the ``DataFrame``.
Use it with ``x='Material'``, ``y='Gth_micro'`` and ``hue='Machine'``, with ``kind='point'``.

**Experiment:** Change the parameter ``kind`` (e.g., ``'bar'``, ``'box'``, ``'violin'``). Which kind of plot shows the differences between the groups best?

In [ ]:
### BEGIN SOLUTION
# Try different kinds, e.g., 'point', 'bar', 'box', 'violin'
sns.catplot(laser, x='Material', y='Gth_micro', hue='Machine', kind='point')
plt.title('Burr Height by Material and Machine')
plt.show()
### END SOLUTION

**Optional:** Numerical attributes can be aggregated into classes (bins) to analyze them like ordinal attributes.

Add a new column ``ThicknessClass`` that assigns ``Thickness_mm`` to the classes ``'<=0.8'``, ``'0.8-1.0'``, ``'1.0-1.2'`` and ``'>1.2'`` (in mm) using ``pd.cut()``.
Then compute a table of the **median** burr height per ``Material`` and ``ThicknessClass`` and visualize it as a bar chart.

**Experiment:** Replace ``median()`` by ``mean()`` and compare both tables. In which groups do they differ most, and why?

In [ ]:
### BEGIN SOLUTION
laser['ThicknessClass'] = pd.cut(laser['Thickness_mm'], bins=[0, 0.8, 1.0, 1.2, 2.0],
                                 labels=['<=0.8', '0.8-1.0', '1.0-1.2', '>1.2'])
gth_material_thickness = laser.groupby(['Material', 'ThicknessClass'], observed=True)['Gth_micro'].median().unstack()
display(gth_material_thickness.round(1))
gth_material_thickness.plot(kind='bar')
plt.ylabel('Median burr height (um)')
plt.legend(title='Thickness (mm)')
plt.show()
### END SOLUTION

In [ ]:
### BEGIN SOLUTION
gth_material_thickness_mean = laser.groupby(['Material', 'ThicknessClass'], observed=True)['Gth_micro'].mean().unstack()
display(gth_material_thickness_mean.round(1))
display(laser.groupby(['Material', 'ThicknessClass'], observed=True)['Gth_micro'].min().unstack().round(2))
### END SOLUTION

#### Explanation

With the median, the burr height increases with the sheet thickness within each material (most clearly for steel above 1.2 mm), and copper has the highest burr heights in every thickness class.

With the mean, some groups are pulled down (e.g., aluminum above 1.2 mm, steel 1.0-1.2 mm): exactly these groups contain the implausible values near 0 µm (see their minimum), which are still in the data. The median is robust against such values, the mean is not.
Note that some groups are small (e.g., copper: 6-11 parts per class).

## 4. Outlier Detection <a id="4-outlier-detection"></a>

[Back to top](#contents)

Find unusual or extreme observations. The previous steps already revealed some suspicious burr heights.

Create box plots (``sns.boxplot()``) of ``Gth_micro`` and ``GasPressure_bar``.

In [ ]:
### BEGIN SOLUTION
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
sns.boxplot(laser, y='Gth_micro', ax=axes[0])
sns.boxplot(laser, y='GasPressure_bar', ax=axes[1])
plt.tight_layout()
plt.show()
### END SOLUTION

Detect outliers in ``Gth_micro`` statistically with both rules from the lecture:

* **IQR rule** with ``k = 1.5``
* **z-score rule** with ``z_thres = 3``

Print the number of outliers found by each rule and display the affected rows sorted by burr height (``sort_values()``).

**Experiment:** Change the thresholds (e.g., ``k = 3``, ``z_thres = 2.5`` or ``3.5``). How does the number of outliers change? Which values are found with all thresholds?

*Hint:* Arithmetic operations on a column are applied to all rows at once, e.g., ``(x - x.mean()) / x.std()`` computes the z-scores of all values of ``x``. Combine conditions with ``|`` (or) and ``&`` (and), as in the pandas tutorial.

In [ ]:
### BEGIN SOLUTION
burr = laser['Gth_micro']
q1, q3 = burr.quantile([0.25, 0.75])
iqr = q3 - q1
k = 1.5
iqr_mask = (burr < q1 - k * iqr) | (burr > q3 + k * iqr)
print('IQR outliers:', iqr_mask.sum())
laser[iqr_mask].sort_values('Gth_micro')
### END SOLUTION

In [ ]:
### BEGIN SOLUTION
z = (burr - burr.mean()) / burr.std()
z_thres = 3
z_mask = z.abs() > z_thres
print('z-score outliers:', z_mask.sum())
laser[z_mask].assign(z=z[z_mask]).sort_values('Gth_micro')
### END SOLUTION

In [ ]:
### BEGIN SOLUTION
for k in [1.5, 3]:
    mask = (burr < q1 - k * iqr) | (burr > q3 + k * iqr)
    print(f'IQR rule, k = {k}: {mask.sum()} outliers:', sorted(burr[mask].round(2)))
for z_thres in [2.5, 3, 3.5]:
    mask = z.abs() > z_thres
    print(f'z-score rule, z_thres = {z_thres}: {mask.sum()} outliers:', sorted(burr[mask].round(2)))
### END SOLUTION

#### Explanation

With the thresholds from the lecture, the IQR rule flags more values than the z-score rule. The additional IQR outliers lie only slightly outside the whiskers and are plausible burr heights.

The larger the threshold, the fewer outliers are found. The four values near 0 µm and the largest burr height (56.4 µm) are found with all thresholds; the other high burr heights (45.8 and 50.3 µm) only with the smaller thresholds. All of these values should be investigated.

Investigate the extreme burr heights in more detail:

* Look at the parts with a burr height **below 1 µm**. Compare the values with ``NeedsDeburring``. What could have happened?
* Look at the parts with a **gas pressure below 4 bar**. How do their burr heights compare to the other parts?

Are these outliers **erroneous** or **valid**? Decide how to handle them and document your decision.

In [ ]:
### BEGIN SOLUTION
display(laser[laser['Gth_micro'] < 1])
display(laser[laser['GasPressure_bar'] < 4])
### END SOLUTION

#### Explanation

* **Burr height < 1 µm:** the values (0.02-0.04) are about 1000 times smaller than all other values, and some of these parts even need deburring. The burr height was most likely recorded in mm instead of µm - **erroneous** outliers, but correctable: multiply by 1000.
* **Gas pressure < 4 bar:** these parts were cut with ~2 bar instead of the usual 6-10 bar (e.g., a gas supply problem). Without enough gas pressure, the molten material is not blown out of the kerf, which explains the high burr heights - **valid** outliers that reveal a real process effect.
  Decision: keep them, but document them. Depending on the modeling goal, they could also be modeled separately (different operating regime).

Correct the unit errors in ``Gth_micro``. Afterwards, recreate the box plot and recompute the Pearson correlations of ``Gth_micro`` with the numerical attributes.

In [ ]:
### BEGIN SOLUTION
pearson_before = laser[cols].corr()['Gth_micro']
unit_error = laser['Gth_micro'] < 1
laser.loc[unit_error, 'Gth_micro'] = laser.loc[unit_error, 'Gth_micro'] * 1000
sns.boxplot(laser, y='Gth_micro')
plt.show()
pd.DataFrame({'before': pearson_before, 'after': laser[cols].corr()['Gth_micro']}).round(2)
### END SOLUTION

## 5. Dealing with High-Dimensional Data <a id="5-dealing-with-high-dimensional-data"></a>

[Back to top](#contents)

With 13 attributes, our dataset is not really high-dimensional, so dimensionality reduction (e.g., PCA) is not necessary.
However, we should still identify **uninformative**, **redundant** and **irrelevant** attributes before modeling.

First, check the number of unique values per attribute (``nunique()``) to find attributes without any variance.

In [ ]:
### BEGIN SOLUTION
laser.nunique()
### END SOLUTION

Next, check how ``NeedsDeburring`` relates to the target: compare the minimum and maximum burr height per class (``groupby()`` and ``agg()``).
Should ``NeedsDeburring`` be used as a feature in a model that predicts ``Gth_micro``?

In [ ]:
### BEGIN SOLUTION
laser.groupby('NeedsDeburring')['Gth_micro'].agg(['min', 'max', 'count'])
### END SOLUTION

#### Explanation

The burr height ranges do not overlap: parts with a burr height above 30 µm need deburring.
``NeedsDeburring`` is derived from the target - using it as a feature would be **target leakage**, as it is only known after the part has been cut and measured.
(It could, however, serve as an alternative target for a classification model.)

Using all your findings so far, create a new ``DataFrame`` ``laser_model`` without the attributes that should not be used as features for a model predicting ``Gth_micro``.
Think about the reason for each dropped attribute.

In [ ]:
### BEGIN SOLUTION
laser_model = laser.drop(columns=['Id', 'Plant', 'NeedsDeburring', 'ProductionTime', 'Temperature_C'])
if 'ThicknessClass' in laser_model.columns:  # only exists if you solved the optional task in Step 3.2
    laser_model = laser_model.drop(columns=['ThicknessClass'])
print(laser_model.shape)
laser_model.head()
### END SOLUTION

#### Explanation

| Dropped attribute | Reason |
|---|---|
| ``Id`` | identifier without information about the process - irrelevant |
| ``Plant`` | constant - uninformative |
| ``NeedsDeburring`` | derived from the target - target leakage |
| ``ProductionTime`` | redundant to ``CuttingSpeed_mm_s`` (Spearman ~ -0.9) and a result of the process rather than a process parameter |
| ``ThicknessClass`` | derived from ``Thickness_mm`` - redundant (only exists if you solved the optional task) |
| ``Temperature_C`` | systematically missing for machine ``m_d_001`` and strongly related to laser power and material (keeping it for a model of machine ``m_si_009`` only is also defensible) |

``LaserPower_W`` and ``CuttingSpeed_mm_s`` are correlated (~0.65), but below the collinearity threshold - keep both.

## 6. Summary and Insights <a id="6-summary-and-insights"></a>

[Back to top](#contents)

Conclude your EDA before modeling. Summarize in a markdown cell:

* the main univariate and multivariate findings,
* the data quality issues you found and your preprocessing decisions (missing values, duplicates, outliers, dropped attributes),
* hypotheses and next steps for modeling.

#### Explanation

**Data quality:**
* 304 rows, 13 attributes; 4 duplicated rows (double-logged parts) - removed.
* Unclear names / units: ``Gth_micro`` (German abbreviation), ``ProductionTime`` (unit unknown).
* ``Temperature_C``: systematically missing for machine ``m_d_001``, sensor dropouts coded as 999 - replaced by ``NaN``; attribute not used as a feature.
* ~2 % randomly missing values in other attributes - rows without target removed, the rest is imputed in the modeling phase.
* 4 burr heights recorded in mm instead of µm - corrected (x 1000).
* 4 parts cut with ~2 bar gas pressure and very high burr - valid outliers, kept.
* ``Plant`` is constant, copper is underrepresented (~12 %).

**Univariate insights:**
* Burr height roughly symmetric around ~25 µm; ``ProductionTime`` is right-skewed.

**Multivariate insights:**
* Burr height increases with sheet thickness and cutting speed and decreases with gas pressure; copper has the highest, aluminum the lowest burr heights. Surface quality and machine show no clear effect.
* Cutting speed increases with laser power (operator setting) - the individual effects of both parameters cannot be separated by EDA alone.
* ``ProductionTime`` is redundant to the cutting speed; ``NeedsDeburring`` is derived from the target.

**Next steps:**
* Features: ``Material``, ``Machine``, ``SurfaceQuality``, ``Thickness_mm``, ``LaserPower_W``, ``CuttingSpeed_mm_s``, ``GasPressure_bar``.
* Impute missing feature values, encode the categorical attributes, scale the numerical ones, and train a first regression model.
* Hypothesis to check with the model: higher cutting speed and lower gas pressure increase the burr height.

---
### EDA Checklist - for Your Own Data

The following checklist summarizes the steps of this exercise. You can use it as a guideline when exploring a new dataset.

| EDA step | Questions to answer | Useful tools (used in this notebook) |
|---|---|---|
| **1. Dataset Understanding** | Which attributes exist? Scale types and units known? Names clear? | ``head()``, ``tail()``, ``info()``, dataset documentation |
| | Plausible value ranges? Coded missing values (e.g., ``999``)? | ``describe()``, ``value_counts()``, ``replace(999, np.nan)`` |
| | Missing values? Duplicate rows? Unique identifiers? | ``isna().sum()``, ``msno.matrix()``, ``duplicated()``, ``drop_duplicates()``, ``is_unique`` |
| | Numeric codes for categories? | ``map()`` with a dictionary |
| **2. Univariate Analysis** | Central tendency, dispersion, skewness? | ``mean()``, ``median()``, ``std()``, ``quantile()``, ``skew()`` |
| | Distributions plausible? Imbalanced categories? | ``sns.countplot()`` (also ``stat='proportion'``), ``sns.histplot()`` (vary ``bins``, ``kde=True``) |
| | Missing values random or systematic? How to handle them? | ``isna().mean()``, ``groupby()`` + share of missing values, ``dropna()`` |
| **3. Bi- & Multivariate Analysis** | Relationships between attributes and with the target? | ``sns.scatterplot()`` (``hue``), ``sns.lmplot()``, ``sns.pairplot()`` |
| | Differences between groups? | ``sns.boxplot()`` (``hue``), ``sns.histplot()`` (``common_norm=False``), ``sns.catplot()``, ``groupby()`` + ``agg()`` / ``unstack()``, ``pd.cut()`` |
| | Linear or monotonic correlations? Collinear attributes (absolute correlation above 0.7)? | ``corr()`` / ``corr(method='spearman')``, ``sns.heatmap()`` |
| **4. Outlier Detection** | Unusual values? Erroneous or valid? Correctable? | ``sns.boxplot()``, IQR rule, z-score rule, ``sort_values()``, filtering, ``.loc[...] = ...`` |
| **5. High-Dimensional Data** | Constant, redundant, or irrelevant attributes? Target leakage? | ``nunique()``, correlation matrix, ``groupby()`` + ``agg(['min', 'max'])``, ``drop()`` |
| **6. Summary and Insights** | Main findings, data quality issues, preprocessing decisions, next steps? | Document everything - your decisions must be traceable! |